# Binary Muon-vs-Other Classification Study

This notebook focuses specifically on the binary classification task: **Muon vs. Non-Muon**.
It implements:
1. Feature engineering of reconstructed-only track properties.
2. Training a binary Random Forest classifier.
3. Generating ROC and Precision-Recall curves.
4. Evaluating efficiency and purity as a function of the muon probability threshold.
5. Comparing performance of baseline cuts vs the Random Forest classifier.
6. Defining working points (High-Efficiency, Balanced, High-Purity).
7. Diagnostic plotting of True Positives, True Negatives, False Positives, and False Negatives.

In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})

print("Imports successful! ✓")

## 1. Load Trees and Compile Reco Features

In [ ]:
file_path = Path("../Cut2.root")
f = uproot.open(file_path)
t_rc = f["Reco_Tree;2"]
t_tr = f["Truth_Info;2"]

rc_arr = t_rc.arrays(["TrackHitPos", "TrackHitEnergies", "nTracks"])
tr_arr = t_tr.arrays([
    "TrueVtxX", "TrueVtxY", "TrueVtxZ", 
    "TrueVtxID", "RecoTrackPrimaryParticleVtxId", 
    "RecoTrackPrimaryParticlePDG"
])
print(f"Loaded {len(rc_arr)} events.")

In [ ]:
def fit_3d_line(hits):
    mean = np.mean(hits, axis=0)
    centered = hits - mean
    cov = np.cov(centered, rowvar=False)
    if cov.ndim < 2:
        return mean, np.array([0.0, 0.0, 1.0]), 0.0
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    direction = eigenvectors[:, np.argmax(eigenvalues)]
    
    diff = centered
    proj = diff - np.outer(diff @ direction, direction)
    residuals = np.sum(proj**2, axis=1)
    mean_residual = np.mean(residuals)
    return mean, direction, mean_residual

def get_dca_and_angle(vtx, p, d, start_hit):
    v_to_p = vtx - p
    dca = np.linalg.norm(v_to_p - (v_to_p @ d) * d)
    v_to_start = start_hit - vtx
    norm = np.linalg.norm(v_to_start)
    if norm > 0:
        cos_angle = (v_to_start @ d) / norm
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * 180.0 / np.pi
    else:
        angle = 0.0
    return dca, angle

In [ ]:
track_records = []

for event_id in range(len(rc_arr)):
    n_tracks = rc_arr["nTracks"][event_id]
    if n_tracks < 1:
        continue
        
    vtx_ids = tr_arr["TrueVtxID"][event_id].tolist()
    track_vtx_ids = tr_arr["RecoTrackPrimaryParticleVtxId"][event_id].tolist()
    track_pdgs = tr_arr["RecoTrackPrimaryParticlePDG"][event_id].tolist()
    
    # Fit tracks
    lines = []
    track_hits = []
    track_energies = []
    
    for track_idx in range(n_tracks):
        hits = ak.to_numpy(rc_arr["TrackHitPos"][event_id][track_idx])
        energies = ak.to_numpy(rc_arr["TrackHitEnergies"][event_id][track_idx])
        mask = hits[:, 0] > -9e8
        actual_hits = hits[mask]
        actual_energies = energies[mask]
        
        if len(actual_hits) >= 2:
            p, d, resid = fit_3d_line(actual_hits)
            lines.append((p, d, resid))
            track_hits.append(actual_hits)
            track_energies.append(actual_energies)
            
    if len(lines) == 0:
        continue
        
    # Reconstruct vertex (reco-only)
    if len(lines) == 1:
        p, d, resid = lines[0]
        actual_hits = track_hits[0]
        reco_vtx = actual_hits[np.argmin(actual_hits[:, 2])]
    else:
        A = np.zeros((3, 3))
        b = np.zeros(3)
        for p, d, resid in lines:
            proj = np.eye(3) - np.outer(d, d)
            A += proj
            b += proj @ p
        try:
            reco_vtx = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            reco_vtx = np.mean([p for p, d, resid in lines], axis=0)
(

In [ ]:
    # Calculate track features
    event_tracks = []
    for track_idx, (p, d, resid) in enumerate(lines):
        actual_hits = track_hits[track_idx]
        actual_energies = track_energies[track_idx]
        
        start_hit = actual_hits[np.argmin(actual_hits[:, 2])]
        end_hit = actual_hits[np.argmax(actual_hits[:, 2])]
        length = np.linalg.norm(end_hit - start_hit)
        nhits = len(actual_hits)
        
        dca, angle = get_dca_and_angle(reco_vtx, p, d, start_hit)
        dist_to_vtx = np.linalg.norm(start_hit - reco_vtx)
        z_depth = np.max(actual_hits[:, 2]) - np.min(actual_hits[:, 2])
        
        total_energy = np.sum(actual_energies)
        dedx = total_energy / length if length > 0 else 0.0
        
        pdg = track_pdgs[track_idx] if track_idx < len(track_pdgs) else 0
        abs_pdg = abs(pdg)
        
        label = "muon" if abs_pdg == 13 else "non-muon"
        is_muon = 1 if abs_pdg == 13 else 0
            
        event_tracks.append({
            "event_id": event_id,
            "track_idx": track_idx,
            "true_pdg": pdg,
            "true_label": label,
            "is_muon": is_muon,
            "length": length,
            "nhits": nhits,
            "dca": dca,
            "angle": angle,
            "dist_to_vtx": dist_to_vtx,
            "z_depth": z_depth,
            "start_z": start_hit[2],
            "end_z": end_hit[2],
            "straightness": resid,
            "total_energy": total_energy,
            "dedx": dedx
        })
        
    # Sort and rank tracks by length
    sorted_by_len = sorted(range(len(event_tracks)), key=lambda k: event_tracks[k]["length"], reverse=True)
    for rank, idx in enumerate(sorted_by_len):
        event_tracks[idx]["length_rank"] = rank + 1
        event_tracks[idx]["is_longest"] = 1 if rank == 0 else 0
        
    # Sort and rank tracks by z_depth
    sorted_by_z = sorted(range(len(event_tracks)), key=lambda k: event_tracks[k]["z_depth"], reverse=True)
    for rank, idx in enumerate(sorted_by_z):
        event_tracks[idx]["z_depth_rank"] = rank + 1
        
    track_records.extend(event_tracks)

df = pd.DataFrame(track_records)
print(f"Compiled {len(df)} track records.")

## 2. Train Binary Random Forest Classifier

In [ ]:
feature_cols = [
    "length", "nhits", "z_depth", "straightness", "total_energy", "dedx",
    "start_z", "end_z", "dca", "angle", "dist_to_vtx", "length_rank", "z_depth_rank", "is_longest"
]
X = df[feature_cols]
y = df["is_muon"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

probs = rf.predict_proba(X_test)[:, 1]
print("Classifier trained successfully!")

## 3. ROC Curve & Precision-Recall Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.05])
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Efficiency)')
axes[0].set_title('Receiver Operating Characteristic')
axes[0].legend(loc="lower right")
axes[0].grid(True, linestyle=':', alpha=0.6)

# PR
axes[1].plot(recalls, precisions, color='blue', lw=2, label='Precision-Recall curve')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('Recall (Efficiency)')
axes[1].set_ylabel('Precision (Purity)')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(loc="lower left")
axes[1].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

## 4. Efficiency and Purity vs. Decision Threshold

In [ ]:
thresholds_extended = np.linspace(0.0, 1.0, 100)
effs = []
purs = []
rejs = []

for th in thresholds_extended:
    preds = (probs >= th).astype(int)
    cm = confusion_matrix(y_test, preds)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        effs.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        purs.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
        rejs.append(tn / (tn + fp) if (tn + fp) > 0 else 0)
    else:
        effs.append(0)
        purs.append(0)
        rejs.append(0)

plt.figure(figsize=(9, 5))
plt.plot(thresholds_extended, effs, label='Muon Efficiency', color='forestgreen', lw=2)
plt.plot(thresholds_extended, purs, label='Muon Purity', color='blue', lw=2)
plt.plot(thresholds_extended, rejs, label='Hadron/Other Rejection', color='red', lw=1.5, linestyle='--')
plt.xlabel('Muon Probability Decision Threshold')
plt.ylabel('Score')
plt.title('Performance vs. Decision Threshold')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 5. Working Points & Baseline Cuts Comparison

- **High-Efficiency WP:** ~90% muon efficiency
- **Balanced WP:** Maximizing F1-score
- **High-Purity WP:** ~90% muon purity

In [ ]:
# Find indices for working points
idx_he = np.argmin(np.abs(np.array(effs) - 0.90))
idx_hp = np.argmin(np.abs(np.array(purs) - 0.90))
f1s = 2 * (np.array(effs) * np.array(purs)) / (np.array(effs) + np.array(purs) + 1e-10)
idx_bal = np.argmax(f1s)

print("=== RECOMMENDED WORKING POINTS ===")
print(f"High-Efficiency WP: Threshold = {thresholds_extended[idx_he]:.3f} | Efficiency = {effs[idx_he]*100:.2f}% | Purity = {purs[idx_he]*100:.2f}% | Rejection = {rejs[idx_he]*100:.2f}%")
print(f"Balanced WP:        Threshold = {thresholds_extended[idx_bal]:.3f} | Efficiency = {effs[idx_bal]*100:.2f}% | Purity = {purs[idx_bal]*100:.2f}% | Rejection = {rejs[idx_bal]*100:.2f}%")
print(f"High-Purity WP:     Threshold = {thresholds_extended[idx_hp]:.3f} | Efficiency = {effs[idx_hp]*100:.2f}% | Purity = {purs[idx_hp]*100:.2f}% | Rejection = {rejs[idx_hp]*100:.2f}%")

In [ ]:
# Baseline Cuts Evaluation
y_pred_baseline = np.where((X_test["length"] > 1800) & (X_test["z_depth"] > 1500), 1, 0)
cm_base = confusion_matrix(y_test, y_pred_baseline)
tn_b, fp_b, fn_b, tp_b = cm_base.ravel()
eff_base = tp_b / (tp_b + fn_b)
pur_base = tp_b / (tp_b + fp_b)
rej_base = tn_b / (tn_b + fp_b)

print("=== BASELINE CUTS COMPARISON ===")
print(f"Baseline cuts:  Efficiency = {eff_base*100:.2f}% | Purity = {pur_base*100:.2f}% | Rejection = {rej_base*100:.2f}%")

# Find RF metrics at equal efficiency
idx_eq_eff = np.argmin(np.abs(np.array(effs) - eff_base))
print(f"RF at equal Eff: Threshold = {thresholds_extended[idx_eq_eff]:.3f} | Efficiency = {effs[idx_eq_eff]*100:.2f}% | Purity = {purs[idx_eq_eff]*100:.2f}% | Rejection = {rejs[idx_eq_eff]*100:.2f}%")

# Find RF metrics at equal purity
idx_eq_pur = np.argmin(np.abs(np.array(purs) - pur_base))
print(f"RF at equal Pur: Threshold = {thresholds_extended[idx_eq_pur]:.3f} | Efficiency = {effs[idx_eq_pur]*100:.2f}% | Purity = {purs[idx_eq_pur]*100:.2f}% | Rejection = {rejs[idx_eq_pur]*100:.2f}%")

## 6. Plotting Feature Distributions for Classification Categories

We categorize the test set into:
- **True Muons (TP):** True = 1, Predicted = 1
- **True Non-Muons (TN):** True = 0, Predicted = 0
- **False Muons (FP):** True = 0, Predicted = 1
- **Missed Muons (FN):** True = 1, Predicted = 0

In [ ]:
df_test = df.loc[X_test.index].copy()
df_test["muon_score"] = probs
# Use balanced threshold
df_test["predicted_label"] = np.where(probs >= thresholds_extended[idx_bal], "muon", "non-muon")

df_test["category"] = "other"
df_test.loc[(df_test["is_muon"] == 1) & (df_test["predicted_label"] == "muon"), "category"] = "TP"
df_test.loc[(df_test["is_muon"] == 0) & (df_test["predicted_label"] == "non-muon"), "category"] = "TN"
df_test.loc[(df_test["is_muon"] == 0) & (df_test["predicted_label"] == "muon"), "category"] = "FP"
df_test.loc[(df_test["is_muon"] == 1) & (df_test["predicted_label"] == "non-muon"), "category"] = "FN"

print("Category counts in test set:")
print(df_test["category"].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features_to_plot = ["length", "nhits", "z_depth", "dedx"]
titles = ["Length (mm)", "Number of Hits", "Z Penetration Depth (mm)", "Energy Loss (dE/dx)"]

colors = {"TP": "blue", "TN": "green", "FP": "red", "FN": "orange"}

for i, feat in enumerate(features_to_plot):
    ax = axes[i // 2][i % 2]
    for cat in ["TP", "TN", "FP", "FN"]:
        sub_df = df_test[df_test["category"] == cat]
        if len(sub_df) > 0:
            ax.hist(sub_df[feat], bins=40, alpha=0.5, label=f"{cat} (N={len(sub_df)})", color=colors[cat], density=True, range=(0, df_test[feat].max() if feat != "dedx" else 0.1))
    ax.set_title(titles[i])
    ax.legend()

plt.tight_layout()
plt.show()

## 7. Save Predictions File

In [ ]:
df_test.to_csv("binary_muon_predictions.csv", index=False)
print("Predictions with all features saved to binary_muon_predictions.csv.")